## Databricks Homework
Since in July 2025 Databricks Community Edition was deprecated and instead of creating separate cluster they are being provided in serverless mode it will be easier for you to work with data - since all the data and tables will be saving not only when cluster as active.

So, no separate activities for cluser creating should be executed - it will be autoattached/started when you will execute any of the cells below.


Please, create table in the default schema using file Sales_December_2019.csv. On the left found Catalog => Add Data => Drop files to upload, or click to browse => Sales_December_2019.csv After file will be uploaded, just need to confirm that table should be uploaded.

 Make sure that the first row is header selected => Create Table. Table will be created with name that you specified (sales_december_2019 by default) You will be able to change the table name later if needed.

PySpark can process SQL queries as a text. In other words you don't need to switch cell language to SQL.
1. Write data from table that you created into the dataframe using PySpark with SQL query. Show data in the dataframe

In [0]:
df = spark.sql("SELECT * FROM workspace.default.sales_december_2019")
df.show()


+--------+--------------------+----------------+----------+--------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  295665|  Macbook Pro Laptop|               1|      1700|12/30/19 00:01|136 Church St, Ne...|
|  295666|  LG Washing Machine|               1|     600.0|12/29/19 07:03|562 2nd St, New Y...|
|  295667|USB-C Charging Cable|               1|     11.95|12/12/19 18:21|277 Main St, New ...|
|  295668|    27in FHD Monitor|               1|    149.99|12/22/19 15:13|410 6th St, San F...|
|  295669|USB-C Charging Cable|               1|     11.95|12/18/19 12:38|43 Hill St, Atlan...|
|  295670|AA Batteries (4-p...|               1|      3.84|12/31/19 22:58|200 Jefferson St,...|
|  295671|USB-C Charging Cable|               1|     11.95|12/16/19 15:10|928 12th St, Port...|
|  295672|USB-C Charging Cable|         

Any notebook can be parameterized using dbutils.widgets. Try to add one parameter "Product_name" and select data from dataframe filtered by value from this parameter. 

2. Select data where product = "product_name" from dataframe using PySpark

In [0]:
dbutils.widgets.text("Product_name", "iPhone")
product_name = dbutils.widgets.get("Product_name")

filtered_df = df.filter(df.Product == product_name)
filtered_df.show()

+--------+-------+----------------+----------+--------------+--------------------+
|Order ID|Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+-------+----------------+----------+--------------+--------------------+
|  295714| iPhone|               1|       700|12/31/19 07:39|826 Hickory St, L...|
|  295726| iPhone|               1|       700|12/25/19 14:49|203 Lakeview St, ...|
|  295735| iPhone|               1|       700|12/22/19 18:25|374 Lincoln St, N...|
|  295737| iPhone|               1|       700|12/19/19 08:51|966 10th St, Atla...|
|  295762| iPhone|               1|       700|12/04/19 08:16|342 9th St, New Y...|
|  295823| iPhone|               1|       700|12/10/19 02:12|852 Maple St, Aus...|
|  295851| iPhone|               1|       700|12/11/19 17:08|197 2nd St, Bosto...|
|  295854| iPhone|               1|       700|12/29/19 10:35|183 Hickory St, A...|
|  295885| iPhone|               1|       700|12/20/19 13:47|223 5th St, Dalla...|
|  2

As well as in SQL, in PySpark you can use aggregate functions. Package pyspark.sql.functions contains all aggregated function from SQL. Try to perform simple aggregation with dataframe. Don't forget, that column types, which you want to calculate, shoud be numerical.  
3. Calculate the sales for each product, including the number of products sold

In [0]:
from pyspark.sql import functions as F

df_typed = df.withColumn("Quantity Ordered", F.expr("try_cast(`Quantity Ordered` as int)")) \
             .withColumn("Price Each", F.expr("try_cast(`Price Each` as double)"))

sales_by_product = df_typed.groupBy("Product").agg(
    F.sum(F.col("Quantity Ordered") * F.col("Price Each")).alias("total_sales"),
    F.sum("Quantity Ordered").alias("total_quantity_sold")
).orderBy(F.col("total_sales").desc())

sales_by_product.show()



+--------------------+------------------+-------------------+
|             Product|       total_sales|total_quantity_sold|
+--------------------+------------------+-------------------+
|  Macbook Pro Laptop|         1094800.0|                644|
|              iPhone|          635600.0|                908|
|     ThinkPad Laptop| 540994.5899999965|                541|
|        Google Phone|          429600.0|                716|
|27in 4K Gaming Mo...| 335781.3899999959|                861|
|34in Ultrawide Mo...|322611.50999999605|                849|
|Apple Airpods Hea...|          311850.0|               2079|
|       Flatscreen TV|          199500.0|                665|
|Bose SoundSport H...|182481.74999999825|               1825|
|    27in FHD Monitor| 144740.3500000011|                965|
|     Vareebadd Phone|          114000.0|                285|
|        20in Monitor| 62804.28999999966|                571|
|            LG Dryer|           51600.0|                 86|
|  LG Wa

In the PySpark you can perform dataframe profiling using one of two special commands or simple aggregated functions. Try to find special commands to complete this task or just use aggregated functions. Hint: please, сhange the column data types based on the data in them

4. Show data profiles output for the new dataframe of table sales_december_2019_csv: row count, min and max value for each column

In [0]:
df_typed.summary("count", "min", "max").show()

+-------+--------+------------+----------------+----------+--------------+--------------------+
|summary|Order ID|     Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+-------+--------+------------+----------------+----------+--------------+--------------------+
|  count|   25037|       25037|           24989|     24989|         25037|               25037|
|    min|  295665|20in Monitor|               1|      2.99|01/01/20 00:10|1 12th St, San Fr...|
|    max|Order ID|      iPhone|               7|    1700.0|    Order Date|    Purchase Address|
+-------+--------+------------+----------------+----------+--------------+--------------------+




5. Add new column to the dataframe from previous task with any default value that you want

In [0]:
df_with_flag = df_typed.withColumn("data_source", F.lit("Sales_December_2019"))
df_with_flag.show()

+--------+--------------------+----------------+----------+--------------+--------------------+-------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|        data_source|
+--------+--------------------+----------------+----------+--------------+--------------------+-------------------+
|  295665|  Macbook Pro Laptop|               1|    1700.0|12/30/19 00:01|136 Church St, Ne...|Sales_December_2019|
|  295666|  LG Washing Machine|               1|     600.0|12/29/19 07:03|562 2nd St, New Y...|Sales_December_2019|
|  295667|USB-C Charging Cable|               1|     11.95|12/12/19 18:21|277 Main St, New ...|Sales_December_2019|
|  295668|    27in FHD Monitor|               1|    149.99|12/22/19 15:13|410 6th St, San F...|Sales_December_2019|
|  295669|USB-C Charging Cable|               1|     11.95|12/18/19 12:38|43 Hill St, Atlan...|Sales_December_2019|
|  295670|AA Batteries (4-p...|               1|      3.84|12/31/19 22:5

Temporary views are processed by cluster and always dropped when the session ends (when the cluster turns off).

6. Create temporary view from task 4 dataframe using PySpark and perform any select using SQL

In [0]:
df_with_flag.createOrReplaceTempView("sales_december_view")

In [0]:
%sql
SELECT Product, `Quantity Ordered`, `Price Each`, data_source
FROM sales_december_view
ORDER BY `Price Each` DESC
LIMIT 10

Product,Quantity Ordered,Price Each,data_source
Macbook Pro Laptop,1,1700.0,Sales_December_2019
Macbook Pro Laptop,1,1700.0,Sales_December_2019
Macbook Pro Laptop,1,1700.0,Sales_December_2019
Macbook Pro Laptop,1,1700.0,Sales_December_2019
Macbook Pro Laptop,1,1700.0,Sales_December_2019
Macbook Pro Laptop,1,1700.0,Sales_December_2019
Macbook Pro Laptop,1,1700.0,Sales_December_2019
Macbook Pro Laptop,1,1700.0,Sales_December_2019
Macbook Pro Laptop,1,1700.0,Sales_December_2019
Macbook Pro Laptop,1,1700.0,Sales_December_2019
